# Predicción de Ventas (Sales Forecasting)

Proyecto para predecir ventas futuras utilizando series temporales con StatsForecast.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Carga de Datos

In [ ]:
df = pd.read_csv('data/sales_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.rename(columns={'date': 'ds', 'sales': 'y'})
df['unique_id'] = 'sales'

print(f"Dataset shape: {df.shape}")
print(f"Rango de fechas: {df['ds'].min()} hasta {df['ds'].max()}")
df.head()

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df['ds'], df['y'], color='steelblue', linewidth=1)
plt.title('Serie Temporal de Ventas', fontsize=14)
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.tight_layout()
plt.show()

In [ ]:
df_eda = df.copy()
df_eda['month'] = df_eda['ds'].dt.month
df_eda['day_of_week'] = df_eda['ds'].dt.dayofweek
df_eda['year'] = df_eda['ds'].dt.year

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

monthly = df_eda.groupby('month')['y'].mean()
axes[0, 0].bar(monthly.index, monthly.values, color='steelblue')
axes[0, 0].set_title('Ventas Promedio por Mes')
axes[0, 0].set_xlabel('Mes')
axes[0, 0].set_ylabel('Ventas Promedio')

dow = df_eda.groupby('day_of_week')['y'].mean()
axes[0, 1].bar(dow.index, dow.values, color='coral')
axes[0, 1].set_title('Ventas Promedio por Día de la Semana')
axes[0, 1].set_xlabel('Día (0=Lunes, 6=Domingo)')
axes[0, 1].set_ylabel('Ventas Promedio')

yearly = df_eda.groupby('year')['y'].mean()
axes[1, 0].bar(yearly.index.astype(str), yearly.values, color='green')
axes[1, 0].set_title('Ventas Promedio por Año')
axes[1, 0].set_xlabel('Año')
axes[1, 0].set_ylabel('Ventas Promedio')

df_eda['y'].hist(bins=30, ax=axes[1, 1], color='purple', alpha=0.7)
axes[1, 1].set_title('Distribución de Ventas')
axes[1, 1].set_xlabel('Ventas')
axes[1, 1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
print("Estadísticas descriptivas de ventas:")
print(df['y'].describe())

## 3. Preparación de Datos

In [ ]:
train_size = int(len(df) * 0.8)
train = df.iloc[:train_size][['ds', 'y', 'unique_id']]
test = df.iloc[train_size:][['ds', 'y', 'unique_id']]

print(f"Training set: {len(train)} días")
print(f"Test set: {len(test)} días")

## 4. Entrenamiento del Modelo StatsForecast

In [ ]:
models = [
    AutoARIMA(season_length=7),
    SeasonalNaive(season_length=7)
]

sf = StatsForecast(models=models, freq='D')
sf.fit(train)

print("Modelo StatsForecast entrenado exitosamente!")

## 5. Predicciones y Evaluación

In [ ]:
forecast = sf.predict(h=len(test), level=[95])
forecast = forecast.reset_index()

predictions = forecast['AutoARIMA'].values
actual = test['y'].values

mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))
mape = mean_absolute_percentage_error(actual, predictions) * 100

print("=" * 50)
print("MÉTRICAS DEL MODELO")
print("=" * 50)
print(f"MAE (Error Absoluto Medio):  ${mae:,.2f}")
print(f"RMSE (Raíz Error Cuadrático Medio): ${rmse:,.2f}")
print(f"MAPE (Error Porcentual Absoluto Medio): {mape:.2f}%")
print("=" * 50)

### Explicación de las Métricas:

- **MAE**: Error promedio en unidades monetarias
- **RMSE**: Penaliza errores grandes más que MAE
- **MAPE**: Error promedio en porcentaje

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train['ds'], train['y'], label='Training', color='steelblue')
plt.plot(test['ds'], test['y'], label='Actual', color='green')
plt.plot(test['ds'], predictions, label='Predicted', color='coral', linestyle='--')
plt.title('Predicciones vs Valores Reales', fontsize=14)
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Predicciones Futuras (30 días)

In [ ]:
full_model = StatsForecast(models=models, freq='D')
full_model.fit(df[['ds', 'y', 'unique_id']])

future_dates = pd.date_range(start=df['ds'].max() + pd.Timedelta(days=1), periods=30, freq='D')
future_df = pd.DataFrame({
    'ds': future_dates,
    'unique_id': ['sales'] * 30
})

forecast_30 = full_model.predict(h=30, level=[95])
forecast_30 = forecast_30.reset_index()

print("Predicciones para los próximos 30 días:")
forecast_30[['ds', 'AutoARIMA', 'AutoARIMA-lo-95', 'AutoARIMA-hi-95']]

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['ds'], df['y'], label='Historico', color='steelblue')
plt.plot(forecast_30['ds'], forecast_30['AutoARIMA'], 
         label='Predicción', color='coral', linewidth=2)
plt.fill_between(forecast_30['ds'], 
                  forecast_30['AutoARIMA-lo-95'], 
                  forecast_30['AutoARIMA-hi-95'], 
                  color='coral', alpha=0.2, label='Intervalo de Confianza 95%')
plt.title('Predicción de Ventas - Próximos 30 días', fontsize=14)
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Guardar Resultados

In [ ]:
import os
os.makedirs('models', exist_ok=True)

forecast_30.to_csv('models/forecast_results.csv', index=False)

print("Resultados guardados en 'models/forecast_results.csv'")